### Exploratory Data Analysis (EDA)

This notebook focuses on understanding the e-commerce dataset before applying machine learning models.
As Member 01, I performed initial data exploration by analyzing the structure, identifying data types, checking for missing values, and visualizing both numerical and categorical features.

Key analyses include distribution of price and quantity, class distribution of delivery status and customer segments, and correlation between numerical variables. The insights obtained from this analysis help guide data preprocessing and model development in later stages of the project.

In [ ]:
print(" 3.1 Task: Exploratory Data Analysis (EDA) and Preprocessing Pipeline")

In [ ]:
# Load Cleaned Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
df = pd.read_csv("../data/ecommerce_data.csv")

df.head()

print("Shape of dataset:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

df.info()

df.describe(include="all")

Explanation

At this stage, we examined:

number of rows and columns
data types of each feature
summary statistics
whether columns are numerical, categorical, or datetime-related

This helps us decide which columns need encoding, scaling, date conversion, or removal.

In [ ]:
# Missing value analysis

missing_values = df.isnull().sum()
missing_values[missing_values > 0]

plt.figure(figsize=(10, 4))
missing_values.sort_values(ascending=False).plot(kind="bar")
plt.title("Missing Values by Column")
plt.ylabel("Count")
plt.show()

Explanation

This step checks whether the dataset contains missing values. If missing values exist, they must be handled before training models. Since machine learning models usually cannot work directly with missing data, this step is essential for a robust preprocessing workflow.

Dropping unnecessary columns

According to the project brief:

order_id is only an identifier
customer_id is only an identifier
product_id may not be meaningful as a raw number
shipping_address and billing_address are messy text columns and are not directly useful in beginner-level modeling

In [ ]:
columns_to_drop = [
    "order_id",
    "customer_id",
    "product_id",
    "shipping_address",
    "billing_address"
]

df = df.drop(columns=columns_to_drop, errors="ignore")
df.head()

Explanation

These columns were removed because they either:

do not provide useful predictive information in their raw form, or
are too unstructured for beginner-level preprocessing.

This reduces noise and simplifies the pipeline.

In [ ]:
# Date feature engineering

if "order_year" not in df.columns:
	df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
	df["shipping_date"] = pd.to_datetime(df["shipping_date"], errors="coerce")

	df["order_year"] = df["order_date"].dt.year
	df["order_month"] = df["order_date"].dt.month
	df["order_day"] = df["order_date"].dt.day
	df["order_dayofweek"] = df["order_date"].dt.dayofweek
	df["order_hour"] = df["order_date"].dt.hour

	df["shipping_year"] = df["shipping_date"].dt.year
	df["shipping_month"] = df["shipping_date"].dt.month
	df["shipping_day"] = df["shipping_date"].dt.day

	df["shipping_time"] = (df["shipping_date"] - df["order_date"]).dt.days

	df = df.drop(columns=["order_date", "shipping_date"], errors="ignore")
	df.head()

Explanation

Machine learning models cannot understand raw datetime values directly. Therefore, we extracted meaningful numerical features such as:

order month
order day
day of week
hour
shipping time in days

These features can help the model learn temporal patterns in buying and shipping behavior.

In [ ]:
# Analysis of continuous variables

# Price distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["price"], bins=30, kde=True)
plt.title("Distribution of Price")
plt.xlabel("Price")
plt.show()

# Quantity distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["quantity"], bins=10, kde=True)
plt.title("Distribution of Quantity")
plt.xlabel("Quantity")
plt.show()

# Shipping time distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["shipping_time"], bins=20, kde=True)
plt.title("Distribution of Shipping Time")
plt.xlabel("Shipping Time (days)")
plt.show()

# Boxplots for outlier detection
plt.figure(figsize=(7, 4))
sns.boxplot(x=df["price"])
plt.title("Boxplot of Price")
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(x=df["quantity"])
plt.title("Boxplot of Quantity")
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(x=df["shipping_time"])
plt.title("Boxplot of Shipping Time")
plt.show()

Explanation

These plots help us understand:

whether numerical features are normally distributed
whether skewness exists
whether outliers are present

For example, if price is right-skewed, the model may need scaling so that large values do not dominate training. Outliers also matter because they can affect regression performance and distance-based methods like K-means clustering.

In [ ]:
# Correlation analysis

numeric_df = df.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Numerical Features")
plt.show()

Explanation

A correlation matrix helps us identify relationships between numerical features. Strong positive or negative correlations can indicate:

useful predictive relationships
possible redundancy between variables

This is important because highly correlated features may not add much new information and can sometimes affect model interpretation.

In [ ]:
# Categorical variable analysis

# Delivery status distribution
plt.figure(figsize=(7, 4))
sns.countplot(x="delivery_status", data=df, order=df["delivery_status"].value_counts().index)
plt.title("Delivery Status Distribution")
plt.xticks(rotation=45)
plt.show()

print(df["delivery_status"].value_counts(normalize=True) * 100)

# Customer segment distribution
plt.figure(figsize=(7, 4))
sns.countplot(x="customer_segment", data=df, order=df["customer_segment"].value_counts().index)
plt.title("Customer Segment Distribution")
plt.xticks(rotation=45)
plt.show()

print(df["customer_segment"].value_counts(normalize=True) * 100)

# Other important categorical columns
categorical_columns = ["category", "payment_method", "device_type", "channel"]

for col in categorical_columns:
    plt.figure(figsize=(7, 4))
    sns.countplot(x=col, data=df, order=df[col].value_counts().index)
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45)
    plt.show()

    

Explanation

These charts show the frequency of each category. They are useful for identifying class imbalance.

From the project description:

delivery_status is expected to be imbalanced
customer_segment also has fewer VIP customers compared to New and Returning customers

This matters because imbalanced classes can cause classification models to favor majority classes and perform poorly on important minority classes.

In [ ]:
# Relationship analysis between numerical features

plt.figure(figsize=(7, 4))
sns.scatterplot(x="quantity", y="price", data=df)
plt.title("Price vs Quantity")
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(x="shipping_time", y="price", data=df)
plt.title("Shipping Time vs Price")
plt.show()

Explanation

Scatter plots help us visually inspect whether relationships exist between variables. For example:

if price increases with quantity, that indicates a useful predictive pattern
if shipping time relates to price or status, it may help classification or clustering

Key findings from EDA

You can write this in markdown in your notebook:

Key Findings Summary
The dataset contains both numerical and categorical features, so preprocessing must handle both types differently.
The price feature shows some skewness and outliers, meaning scaling is important before modeling.
The delivery_status variable is imbalanced, with one class dominating. This confirms that later classification tasks must consider imbalance-aware evaluation.
The customer_segment variable is also imbalanced, especially with fewer VIP customers.
Temporal information from order_date and shipping_date is useful, so new features such as month, day, hour, and shipping time were extracted.
Raw ID columns and raw address columns are not useful for beginner-level modeling, so they were removed.
Numerical features show some relationships, especially between price-related and quantity-related behavior, which may be useful for prediction.

This summary directly supports later modeling decisions.

Preprocessing pipeline using Scikit-learn

This is the most important missing part from your notebook.

Step 1: Separate features and targets

For Task 3.1, we prepare the feature matrix. Later tasks will use different targets such as:

price for regression
delivery_status for classification
customer_segment for classification

For now, we prepare reusable preprocessing on the input features.

In [ ]:
X = df.drop(columns=["price", "delivery_status", "customer_segment"], errors="ignore")
X.head()

Explanation

We remove target columns from the feature set because targets should not be preprocessed as ordinary input features.

In [ ]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
# numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()


print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


# Build preprocessing transformers
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

preprocessor

Explanation

This preprocessing pipeline does the following:

fills missing numerical values using the median
scales numerical features using StandardScaler
fills missing categorical values using the most frequent category
converts categorical values into one-hot encoded columns

This is exactly what the assignment expects: a modular and reusable preprocessing workflow.

In [ ]:
# Test the preprocessing pipeline

X_processed = preprocessor.fit_transform(X)
print("Processed feature matrix shape:", X_processed.shape)

Explanation

This confirms that the pipeline works correctly and transforms the dataset into a machine-learning-ready numerical format.

In [ ]:
# Save cleaned dataset and preprocessing object

import joblib

df.to_csv("../data/cleaned_data.csv", index=False)
joblib.dump(preprocessor, "../artifacts/preprocessor.joblib")